# 02 — House PDF download & manifest

## Objectif

Télécharger les PDF PTR House listés dans `house_ptr_index.csv`.

Output principal : `data/audit/house_pdf_manifest.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.house_download import download_all_pdfs, summarize_manifest, write_download_report
from src.utils import PROCESSED_HOUSE_DIR, AUDIT_DIR, RAW_HOUSE_PDF_DIR, require_file

print("Imports OK")

Imports OK


In [2]:
TARGET_YEAR = 2025
SLEEP_SECONDS = 0.25
FORCE_DOWNLOAD = False
MAX_FILES = None
SAVE_EVERY = 100

ptr_index_path = PROCESSED_HOUSE_DIR / "house_ptr_index.csv"
manifest_path = AUDIT_DIR / "house_pdf_manifest.csv"

In [3]:
require_file(ptr_index_path, "Run 01_house_index_audit.ipynb first.")
df_ptr = pd.read_csv(ptr_index_path, dtype={"doc_id": str})
df_ptr = df_ptr[df_ptr["year"] == TARGET_YEAR].copy().reset_index(drop=True)

print(f"df_ptr shape (year={TARGET_YEAR}):", df_ptr.shape)
display(df_ptr.head())

df_ptr shape (year=2025): (515, 11)


,year,filing_type,filing_date_raw,filing_date,doc_id,last_name,first_name,declarant_name_raw,state_dst,pdf_url,source_xml_path
0,2025,P,9/10/2025,2025-09-10,20032062,Aderholt,Robert B.,Robert B. Aderholt,AL04,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
1,2025,P,1/16/2025,2025-01-16,20026537,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
2,2025,P,2/20/2025,2025-02-20,20026727,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
3,2025,P,3/12/2025,2025-03-12,20027927,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
4,2025,P,5/15/2025,2025-05-15,20030316,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...


## Téléchargement

Le downloader est idempotent : un PDF existant et valide est conservé.

In [4]:
manifest = download_all_pdfs(
    df_ptr,
    output_base_dir=RAW_HOUSE_PDF_DIR,
    manifest_path=manifest_path,
    sleep_seconds=SLEEP_SECONDS,
    force=FORCE_DOWNLOAD,
    max_files=MAX_FILES,
    save_every=SAVE_EVERY,
)

print("manifest shape:", manifest.shape)
display(manifest.head())

House PTR PDFs:   0%|          | 0/515 [00:00<?, ?it/s]

manifest shape: (515, 16)


,year,doc_id,filing_date,declarant_name_raw,state_dst,pdf_url,local_path,http_status,download_status,file_size_bytes,sha256,content_type,n_pages,text_extractable_flag,downloaded_at,error_message
0,2025,20032062,2025-09-10,Robert B. Aderholt,AL04,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,200,ok,62508,e70be3e0b326c459adbfee5b8f232905f203760887cb91...,application/pdf,1,True,2026-06-23T16:04:04+00:00,
1,2025,20026537,2025-01-16,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,200,ok,88771,84bd26667d4ffb8e956be439243bf08aa51f9a368df905...,application/pdf,2,True,2026-06-23T16:04:04+00:00,
2,2025,20026727,2025-02-20,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,200,ok,89319,e034208f56244c403d1b4c47dfb41b3c1b08be65cc2421...,application/pdf,2,True,2026-06-23T16:04:05+00:00,
3,2025,20027927,2025-03-12,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,200,ok,65960,decda87eb1992b3468a72e2df95d1667257bce77630b2f...,application/pdf,1,True,2026-06-23T16:04:06+00:00,
4,2025,20030316,2025-05-15,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...,200,ok,88911,f5bd7087864333b5cf9436465f12e0b9153a30fda9b493...,application/pdf,2,True,2026-06-23T16:04:06+00:00,


In [5]:
summary = summarize_manifest(manifest, expected_count=len(df_ptr) if MAX_FILES is None else MAX_FILES)
summary

{'timestamp': '2026-06-23T16:08:27+00:00',
 'expected_pdf_count': 515,
 'manifest_rows': 515,
 'downloaded_or_existing_count': 515,
 'missing_count': 0,
 'invalid_pdf_count': 0,
 'zero_byte_count': 0,
 'exception_count': 0,
 'http_error_count': 0,
 'success_rate': 1.0,
 'status_counts': {'ok': 515},
 'problems_by_year': {}}

In [6]:
if not manifest.empty:
    display(manifest["download_status"].value_counts(dropna=False))
    problems = manifest[~manifest["download_status"].isin(["ok", "skipped_existing"])]
    display(problems.head(20))

download_status
ok    515
Name: count, dtype: int64

,year,doc_id,filing_date,declarant_name_raw,state_dst,pdf_url,local_path,http_status,download_status,file_size_bytes,sha256,content_type,n_pages,text_extractable_flag,downloaded_at,error_message


In [7]:
report_path = write_download_report(summary, manifest_path=manifest_path)
print("Written:", manifest_path.relative_to(ROOT))
print("Written:", report_path.relative_to(ROOT))

Written: data/audit/house_pdf_manifest.csv
Written: reports/house_download_audit.md


## Conclusion

**OK** si le manifest explique les succès, skips et échecs.

**NEXT** — auditer la qualité texte avec `03_house_pdf_quality_smoke_test.ipynb`.